# 06.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/04-hierarchical/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
names = list("ABCDE")
points = np.array([[1, 1], [2, 1], [5, 4], [6, 5], [6, 8]], dtype=float)   # (orders, items per order)

## Exercise 1: complete linkage by hand

Using the distance matrix from the README, run **complete** linkage on the five customers by hand. Write down each merge and its height. Then check with `linkage(points, method="complete")`.

In [ ]:
# your code here

## Exercise 2: the bridge

Build two round blobs of 100 points each, centred at (0, 0) and (6, 0), with a thin line of 15 points between them:

```python
rng = np.random.default_rng(0)
blob1 = rng.normal([0, 0], 0.7, size=(100, 2))
blob2 = rng.normal([6, 0], 0.7, size=(100, 2))
bridge = np.column_stack([np.linspace(1, 5, 15), np.zeros(15)])
pts = np.vstack([blob1, blob2, bridge])
```

Cluster into 2 groups with single linkage and with Ward. Plot both. Which separates the blobs?

In [ ]:
# your code here

## Exercise 3: do the clusters nest?

On the 300-customer sample, cut the Ward tree into 4 and into 5 clusters. Check that every 5-cluster group sits inside a single 4-cluster group (a crosstab helps). Then do the same with K-Means for $k$ = 4 and 5.

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

sample = rfm.sample(300, random_state=42)
Xs = StandardScaler().fit_transform(np.log1p(sample))
print("sample:", Xs.shape)

In [ ]:
# your code here